# 1. Install & Import Libraries
**Ensemble: ConvNeXt-Small (Exp 7) + EfficientNet-B3 (Exp 5)**

Target: break 80% accuracy by combining both models' complementary per-class strengths.

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install transformers scikit-learn pillow pandas numpy matplotlib huggingface_hub ipywidgets opencv-python google-cloud-storage

# 2. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast
from torchvision import transforms
from transformers import (EfficientNetForImageClassification, ConvNextForImageClassification,
                          AutoImageProcessor)
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import os, json
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)
from google.cloud import storage

# 3. Download Dataset From Storage Bucket

In [ ]:
client = storage.Client()
bucket = client.bucket('naic-dataset-images')
os.makedirs('dataset/image', exist_ok=True)

print('Fetching labels.csv...')
bucket.blob('labels.csv').download_to_filename('dataset/labels.csv')
print('labels.csv saved.')

blobs = bucket.list_blobs(prefix='image/')
download_count = 0
for i, blob in enumerate(blobs):
    if not blob.name.endswith('/'):
        filename = blob.name.split('/')[-1]
        local_path = f'dataset/image/{filename}'
        if not os.path.exists(local_path):
            blob.download_to_filename(local_path)
            download_count += 1
    if i % 500 == 0:
        print(f'Scanned {i} files...')
print(f'Download complete! Fetched {download_count} new images.')

# 4. Read CSV & Stratified Test Split
Same 80/20 split with `random_state=42` as used during training — guarantees the test set is identical to what the models never saw during training.

In [ ]:
df = pd.read_csv('dataset/labels.csv')
train_val_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['Label'], random_state=42
)
print(f'Test set size: {len(test_df)}')
print(f'Class distribution in test set:')
print(test_df['Label'].value_counts().sort_index())

# 5. CLAHE Preprocessing + Retina Cropping
Same pipeline as training — Green-channel CLAHE (clipLimit=2.0) + auto-crop black borders.

In [ ]:
def crop_fundus(image, tol=10):
    """Remove black borders around circular fundus image."""
    if isinstance(image, Image.Image):
        image = np.array(image)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    coords = np.argwhere(mask)
    if coords.size == 0:
        return image
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    return image[y0:y1, x0:x1]

def preprocess_fundus(image):
    """Auto-crop + Green Channel CLAHE pipeline."""
    cropped = crop_fundus(image)
    r, g, b = cv2.split(cropped)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g_enhanced = clahe.apply(g)
    enhanced = cv2.merge((r, g_enhanced, b))
    return Image.fromarray(enhanced)

# 6. Custom Dataset Loader

In [ ]:
class MedicalDatasetLoader(Dataset):
    def __init__(self, df, img_dir, transform=None, use_preprocessing=True):
        self.data = df
        self.img_dir = img_dir
        self.transform = transform
        self.use_preprocessing = use_preprocessing

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['Image']
        label    = int(self.data.iloc[idx]['Label'])
        img_path = os.path.join(self.img_dir, img_name)
        image    = Image.open(img_path).convert('RGB')
        if self.use_preprocessing:
            image = preprocess_fundus(image)
        if self.transform:
            image = self.transform(image)
        return image, label

# 7. Setup CUDA Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# 8. Attention Modules (CBAM + ECA)
**Identical** to the training wrappers. These must match the checkpoint structure exactly, otherwise the weights won't load.

- **ECA**: channel attention with adaptive 1D conv (no dim reduction) — used by both best models
- **CBAM**: channel + spatial attention — available if you need it for other experiment variants

In [ ]:
import math


# ── ECA: Efficient Channel Attention (Wang et al., 2020) ─────────────────────
class ECA(nn.Module):
    """Adaptive kernel size: k = |log2(C)/gamma + b/gamma|, forced odd."""
    def __init__(self, in_channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(in_channels)) / gamma + b / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv     = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(1, 2)
        y = self.conv(y)
        y = self.sigmoid(y)
        y = y.transpose(1, 2).unsqueeze(-1)
        return x * y


# ── CBAM: Channel + Spatial (Woo et al., 2018) ───────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(combined))


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(in_channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x


print('ECA, CBAM attention modules defined.')

# 9. Model Architecture Wrappers
**CRITICAL**: These wrappers must exactly match the training-time architecture for checkpoints to load correctly.

- `EfficientNetWithECA` / `EfficientNetWithCBAM` — from `exp_7_efficienetb3_*.ipynb`
- `ConvNextWithECA` / `ConvNextWithCBAM` — from `convnext_best.ipynb`

Both use the HuggingFace `transformers` backbone (not `torchvision.models`). **Using the wrong backbone would silently load random weights.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EfficientNet-B3 Wrappers — matches exp_5/exp_7 training architecture
# ─────────────────────────────────────────────────────────────────────────────
class EfficientNetWithECA(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.efficientnet = base_model.efficientnet
        feature_dim       = base_model.classifier.in_features
        self.eca          = ECA(feature_dim)
        self.dropout      = base_model.dropout
        self.classifier   = base_model.classifier

    def forward(self, pixel_values):
        features = self.efficientnet(pixel_values).last_hidden_state
        attended = self.eca(features)
        pooled   = attended.mean(dim=[-2, -1])
        pooled   = self.dropout(pooled)
        return self.classifier(pooled)


class EfficientNetWithCBAM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.efficientnet = base_model.efficientnet
        feature_dim       = base_model.classifier.in_features
        self.cbam         = CBAM(feature_dim)
        self.dropout      = base_model.dropout
        self.classifier   = base_model.classifier

    def forward(self, pixel_values):
        features = self.efficientnet(pixel_values).last_hidden_state
        attended = self.cbam(features)
        pooled   = attended.mean(dim=[-2, -1])
        pooled   = self.dropout(pooled)
        return self.classifier(pooled)


# ─────────────────────────────────────────────────────────────────────────────
# ConvNeXt-Small Wrappers — matches convnext_best training architecture
# ─────────────────────────────────────────────────────────────────────────────
class ConvNextWithECA(nn.Module):
    def __init__(self, base_model, feature_dim):
        super().__init__()
        self.convnext   = base_model.convnext
        self.eca        = ECA(feature_dim)
        self.classifier = base_model.classifier

    def forward(self, pixel_values):
        features = self.convnext(pixel_values).last_hidden_state
        attended = self.eca(features)
        pooled   = attended.mean(dim=[-2, -1])
        return self.classifier(pooled)


class ConvNextWithCBAM(nn.Module):
    def __init__(self, base_model, feature_dim):
        super().__init__()
        self.convnext   = base_model.convnext
        self.cbam       = CBAM(feature_dim)
        self.classifier = base_model.classifier

    def forward(self, pixel_values):
        features = self.convnext(pixel_values).last_hidden_state
        attended = self.cbam(features)
        pooled   = attended.mean(dim=[-2, -1])
        return self.classifier(pooled)


print('EfficientNet and ConvNeXt wrappers defined.')

# 10. ⚙️ ENSEMBLE FLAGS & CONFIGURATION — Edit This Cell

Toggle-style configuration matching Cell 14 of your training notebooks. Change flags to switch strategy, model inclusion, attention types, and checkpoint paths.

### Strategy options (from `ensemble_strategies.md`)

| Strategy | What it does | When to use |
|---|---|---|
| `simple_avg` | 50/50 softmax average | Baseline — always run first |
| `weighted_avg` | Tunable model weights | If one model clearly better overall |
| `per_class_weighted` | Class-specific weights | Best when per-class strengths differ |
| `majority_vote` | Hard-label vote | Maximize accuracy, lose minority recall |
| `rank_fusion` | Borda count (rank-based) | Robust to calibration differences |
| `all` | Run all 5, compare | Final sweep for report |

### Model toggles
Set `use_efficientnet` or `use_convnext` to `False` to drop that model from the ensemble. At least one must be `True`.

In [ ]:
# ================================================================================
# ENSEMBLE FEATURE FLAGS — toggle each component ON/OFF
# ================================================================================

ENSEMBLE_FLAGS = {
    # ── Strategy selection (must match one of strategy_map keys, or 'all') ──
    'strategy':              'all',  # simple_avg | weighted_avg | per_class_weighted | majority_vote | rank_fusion | all
    'use_tta':               True,   
    'use_efficientnet':      True,   
    'use_convnext':          True,   
    'effnet_attention':      'eca',  # 'eca' | 'cbam' | 'none' — Exp 5 used 'eca'
    'convnext_attention':    'eca',  # 'eca' | 'cbam' | 'none' — Exp 7 used 'eca'

    # ConvNeXt weighted higher — slightly better overall accuracy + recall
    'convnext_weight':       0.6,
    'efficientnet_weight':   0.4,

    # ── Per-class weights (used by 'per_class_weighted') ────────────────────
    'per_class_weights': {
        'convnext':     [0.40, 0.65, 0.50, 0.60, 0.55],  # ConvNeXt stronger on C1/C3/C4
        'efficientnet': [0.60, 0.35, 0.50, 0.40, 0.45],  # EffNet stronger on C0
    },
}

# ================================================================================
# CHECKPOINT PATHS — UPDATE THESE to your actual experiment directories
# ================================================================================

MODEL_PATHS = {
    'efficientnet': {
        'hf_model_name':   'google/efficientnet-b3',
        'weights_dir':     'efficientnetb3_best_weights/',  # UPDATE
        'weight_pattern':  'best_model_fold_{fold}.pth',
        'img_size':        300,
        'num_folds':       5,
    },
    'convnext': {
        'hf_model_name':   'facebook/convnext-small-224',
        'weights_dir':     'convnext_best_weights/',       # UPDATE
        'weight_pattern':  'best_model_fold_{fold}.pth',
        'img_size':        224,
        'feature_dim':     768,  # ConvNeXt-Small last hidden dim
        'num_folds':       5,
    },
}

# ─── Validation ──────────────────────────────────────────────────────────────
assert ENSEMBLE_FLAGS['use_efficientnet'] or ENSEMBLE_FLAGS['use_convnext'], \
    'At least one model must be enabled.'
assert ENSEMBLE_FLAGS['effnet_attention'] in ('eca', 'cbam', 'none')
assert ENSEMBLE_FLAGS['convnext_attention'] in ('eca', 'cbam', 'none')
valid_strategies = {'simple_avg', 'weighted_avg', 'per_class_weighted',
                    'majority_vote', 'rank_fusion', 'all'}
assert ENSEMBLE_FLAGS['strategy'] in valid_strategies, \
    f'Strategy must be one of: {valid_strategies}'

# Weight sanity checks
if ENSEMBLE_FLAGS['strategy'] in ('weighted_avg', 'all'):
    w_sum = ENSEMBLE_FLAGS['convnext_weight'] + ENSEMBLE_FLAGS['efficientnet_weight']
    assert abs(w_sum - 1.0) < 1e-6, f'Global weights must sum to 1.0, got {w_sum}'

if ENSEMBLE_FLAGS['strategy'] in ('per_class_weighted', 'all'):
    for c in range(5):
        pc_sum = (ENSEMBLE_FLAGS['per_class_weights']['convnext'][c] +
                  ENSEMBLE_FLAGS['per_class_weights']['efficientnet'][c])
        assert abs(pc_sum - 1.0) < 1e-6, f'Per-class weights for C{c} must sum to 1.0, got {pc_sum}'

# ─── Print active config ─────────────────────────────────────────────────────
print('=' * 60)
print('ENSEMBLE CONFIGURATION')
print('=' * 60)
print(f"  Strategy:          {ENSEMBLE_FLAGS['strategy']}")
print(f"  TTA (8-view):      {'ON' if ENSEMBLE_FLAGS['use_tta'] else 'OFF'}")
print(f"  EfficientNet-B3:   {'ON' if ENSEMBLE_FLAGS['use_efficientnet'] else 'OFF'} "
      f"(attention: {ENSEMBLE_FLAGS['effnet_attention']})")
print(f"  ConvNeXt-Small:    {'ON' if ENSEMBLE_FLAGS['use_convnext'] else 'OFF'} "
      f"(attention: {ENSEMBLE_FLAGS['convnext_attention']})")
print('=' * 60)

# 11. TTA Transforms (8 views per model)
Same 8 deterministic transforms as the training-time TTA. Each model has its own set scaled to its input resolution (EffNet: 300×300, ConvNeXt: 224×224) and its own normalization constants from its HuggingFace processor.

In [ ]:
# Load processors for correct normalization constants per model
effnet_processor   = AutoImageProcessor.from_pretrained(MODEL_PATHS['efficientnet']['hf_model_name'])
convnext_processor = AutoImageProcessor.from_pretrained(MODEL_PATHS['convnext']['hf_model_name'])

EFFNET_MEAN   = effnet_processor.image_mean
EFFNET_STD    = effnet_processor.image_std
CONVNEXT_MEAN = convnext_processor.image_mean
CONVNEXT_STD  = convnext_processor.image_std


def build_tta_transforms(img_size, mean, std):
    """8 deterministic augmented views — matches training-time TTA."""
    norm = transforms.Normalize(mean=mean, std=std)
    base = lambda *ops: transforms.Compose(
        [transforms.Resize((img_size, img_size)), *ops, transforms.ToTensor(), norm]
    )
    return [
        base(),                                                                       # 1. identity
        base(transforms.RandomHorizontalFlip(p=1.0)),                                # 2. H-flip
        base(transforms.RandomVerticalFlip(p=1.0)),                                  # 3. V-flip
        base(transforms.RandomRotation(degrees=(10, 10))),                           # 4. +10 rotation
        base(transforms.RandomRotation(degrees=(-10, -10))),                         # 5. -10 rotation
        base(transforms.ColorJitter(brightness=0.15)),                               # 6. brightness
        base(transforms.RandomHorizontalFlip(p=1.0),                                 # 7. H-flip + rot
             transforms.RandomRotation(degrees=(10, 10))),
        base(transforms.RandomVerticalFlip(p=1.0),                                   # 8. V-flip + rot
             transforms.RandomRotation(degrees=(-10, -10))),
    ]


effnet_tta   = build_tta_transforms(MODEL_PATHS['efficientnet']['img_size'], EFFNET_MEAN,   EFFNET_STD)
convnext_tta = build_tta_transforms(MODEL_PATHS['convnext']['img_size'],     CONVNEXT_MEAN, CONVNEXT_STD)

assert len(effnet_tta) == 8 and len(convnext_tta) == 8
print(f'EfficientNet TTA: {len(effnet_tta)} views @ {MODEL_PATHS["efficientnet"]["img_size"]}x{MODEL_PATHS["efficientnet"]["img_size"]}')
print(f'ConvNeXt TTA:     {len(convnext_tta)} views @ {MODEL_PATHS["convnext"]["img_size"]}x{MODEL_PATHS["convnext"]["img_size"]}')

# 12. Unified Inference Engine

**What this cell does**: For each model (EffNet, ConvNeXt), loads every fold checkpoint, runs 8-view TTA on the test set, and returns a single `(N, 5)` probability matrix per model.

**Full data path**:
```
Per fold:      8 TTA views   →  average  →  1 fold prob matrix
Per model:     5 fold probs  →  average  →  1 model prob matrix   ← 5 weights blended
Both models:   2 model probs →  strategy →  final predictions    ← 10 weights total
```

Each test image runs through **80 forward passes** (5 folds × 8 views × 2 models) before the ensemble strategy is applied.

**Three performance optimizations baked in**:
1. **CLAHE+crop cached once per model** (`CachedPreprocessedDataset`) — without this, the same image is re-preprocessed 80 times. With it, preprocessing runs exactly twice (once per model).
2. **DataLoader reused across TTA views** — only the augmentation transform is swapped, not the loader or the dataset.
3. **Optional per-fold weighting** — if post-training analysis shows Fold 3 is dramatically better than Fold 0, pass `fold_weights={0: 0.5, 1: 1.0, 2: 1.0, 3: 2.0, 4: 1.0}` to down-weight the weak fold. Defaults to equal weights if `None`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODEL BUILDER
# ─────────────────────────────────────────────────────────────────────────────
def build_model(model_type, attention, device):
    """
    Construct the correct architecture based on model_type + attention flag.
    Returns (model, is_wrapped) — is_wrapped affects how forward() is called.
    """
    cfg = MODEL_PATHS[model_type]

    if model_type == 'efficientnet':
        base = EfficientNetForImageClassification.from_pretrained(
            cfg['hf_model_name'], num_labels=5, ignore_mismatched_sizes=True
        ).to(device)
        if attention == 'eca':
            model = EfficientNetWithECA(base).to(device)
        elif attention == 'cbam':
            model = EfficientNetWithCBAM(base).to(device)
        else:
            model = base
    elif model_type == 'convnext':
        base = ConvNextForImageClassification.from_pretrained(
            cfg['hf_model_name'], num_labels=5, ignore_mismatched_sizes=True
        ).to(device)
        feature_dim = cfg['feature_dim']
        if attention == 'eca':
            model = ConvNextWithECA(base, feature_dim).to(device)
        elif attention == 'cbam':
            model = ConvNextWithCBAM(base, feature_dim).to(device)
        else:
            model = base
    else:
        raise ValueError(f'Unknown model_type: {model_type}')

    is_wrapped = (attention != 'none')
    return model, is_wrapped


def load_state_dict_flexible(model, ckpt_path, device):
    """
    Load weights from either a plain state_dict or a full checkpoint dict.
    Warns loudly on large key mismatches — strict=False can silently load garbage.
    """
    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state_dict = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        state_dict = ckpt['state_dict']
    else:
        state_dict = ckpt

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'      [!] {len(missing)} missing keys (first 3: {missing[:3]})')
    if unexpected:
        print(f'      [!] {len(unexpected)} unexpected keys (first 3: {unexpected[:3]})')
    if len(missing) > 5 or len(unexpected) > 5:
        print('      [!] WARNING: Large key mismatch — architecture may not match checkpoint!')
    return model


# ─────────────────────────────────────────────────────────────────────────────
# TTA DATASET CACHE
# ─────────────────────────────────────────────────────────────────────────────
# CLAHE + fundus crop is expensive (opencv ops). Without caching, it runs
# 80 times per image (5 folds × 8 views × 2 models = 80 passes of the dataset).
# By pre-computing the CLAHE-preprocessed PIL images ONCE, we save ~75% of the
# preprocessing wall time while keeping the TTA augmentations themselves fresh
# (flips, rotations, etc. are cheap and still applied per-view).
# ─────────────────────────────────────────────────────────────────────────────

class CachedPreprocessedDataset(Dataset):
    """
    Caches CLAHE+crop output PIL images in RAM on first access, then applies
    the TTA transform on top. The transform is swappable per TTA view without
    re-running CLAHE.
    """
    def __init__(self, df, img_dir, transform=None):
        self.data = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self._cache = {}   # idx -> preprocessed PIL.Image

    def set_transform(self, transform):
        """Swap the augmentation transform without invalidating the CLAHE cache."""
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if idx not in self._cache:
            img_name = self.data.iloc[idx]['Image']
            img_path = os.path.join(self.img_dir, img_name)
            image = Image.open(img_path).convert('RGB')
            image = preprocess_fundus(image)   # CLAHE + crop — cached
            self._cache[idx] = image
        image = self._cache[idx]
        label = int(self.data.iloc[idx]['Label'])
        if self.transform:
            image = self.transform(image)
        return image, label


# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE ENGINE
# Per model: loads 5 folds × runs 8 TTA views × averages across all.
# Returns (N, 5) probability matrix.
# ─────────────────────────────────────────────────────────────────────────────

def run_inference_for_model(model_type, attention, test_df, tta_transforms,
                            use_tta=True, fold_weights=None):
    """
    Snapshot-ensemble + TTA for one model family.

    Args:
        model_type     : 'efficientnet' or 'convnext'
        attention      : 'eca', 'cbam', or 'none' — must match training config
        test_df        : pandas df with 'Image' and 'Label' columns
        tta_transforms : list of 8 transforms (or 1 if use_tta=False)
        use_tta        : if False, only uses the identity transform
        fold_weights   : optional dict {fold_idx: weight} to weight folds
                         differently when averaging. If None, equal-weight mean.

    Returns:
        Tensor of shape (N_samples, 5) — averaged probabilities
    """
    cfg            = MODEL_PATHS[model_type]
    weights_dir    = cfg['weights_dir']
    weight_pattern = cfg['weight_pattern']
    num_folds      = cfg['num_folds']

    views_to_run = tta_transforms if use_tta else [tta_transforms[0]]
    n_views      = len(views_to_run)

    # ── Build the CLAHE cache ONCE per model ──────────────────────────────────
    # One dataset instance, one loader spawn, cache persists across all
    # 5 folds × 8 views for this model.
    print(f'  Initializing CLAHE-cached dataset for {model_type}...')
    cached_dataset = CachedPreprocessedDataset(test_df, 'dataset/image', transform=views_to_run[0])

    # Warm the cache with a single sequential pass (num_workers=0 so cache
    # populates in the main process, not forked subprocess memory).
    warmup_loader = DataLoader(cached_dataset, batch_size=32, shuffle=False,
                               num_workers=0, pin_memory=False)
    for _ in warmup_loader:
        pass
    print(f'  Cache warmed: {len(cached_dataset._cache)} preprocessed images ready.')

    # ── Inference loop ────────────────────────────────────────────────────────
    fold_probs_list = []
    loaded_folds    = []

    for fold in range(num_folds):
        model_path = os.path.join(weights_dir, weight_pattern.format(fold=fold))
        if not os.path.exists(model_path):
            print(f'  [fold {fold}] SKIP — not found: {model_path}')
            continue

        print(f'  [fold {fold}] loading {os.path.basename(model_path)}')

        print(f'  🚨 ACTUAL FULL PATH BEING LOADED: {model_path}')
        
        model, is_wrapped = build_model(model_type, attention, device)
        model = load_state_dict_flexible(model, model_path, device)
        model.eval()

        view_probs_for_fold = []
        for v_idx, tta_transform in enumerate(views_to_run):
            # Swap the TTA transform — CLAHE cache stays intact
            cached_dataset.set_transform(tta_transform)
            loader = DataLoader(cached_dataset, batch_size=16, shuffle=False,
                                num_workers=0, pin_memory=True)

            batch_probs = []
            with torch.no_grad():
                for inputs, _ in loader:
                    inputs = inputs.to(device)
                    with autocast(device_type='cuda'):
                        outputs = (model(inputs) if is_wrapped
                                   else model(pixel_values=inputs).logits)
                    probs = F.softmax(outputs.float(), dim=1)
                    batch_probs.append(probs.cpu())
            view_probs_for_fold.append(torch.cat(batch_probs, dim=0))
            if use_tta:
                print(f'    view {v_idx+1}/{n_views} done')

        # Average TTA views within this fold
        fold_avg = torch.stack(view_probs_for_fold).mean(dim=0)
        fold_probs_list.append(fold_avg)
        loaded_folds.append(fold)

        del model
        torch.cuda.empty_cache()

    if not fold_probs_list:
        raise RuntimeError(f'No weights loaded for {model_type} — check MODEL_PATHS.')

    # ── Cross-fold aggregation ────────────────────────────────────────────────
    if fold_weights is None:
        # Equal-weight mean across all loaded folds
        print(f'  Averaging across {len(loaded_folds)} fold(s): {loaded_folds} (equal weights)')
        final_probs = torch.stack(fold_probs_list).mean(dim=0)
    else:
        # Weighted sum — useful if some folds are clearly stronger than others
        weights = torch.tensor([fold_weights.get(f, 0.0) for f in loaded_folds])
        if weights.sum() <= 0:
            raise ValueError(f'fold_weights sum to 0 for {model_type} — check config.')
        weights = weights / weights.sum()                # normalize to sum to 1
        print(f'  Weighted avg over folds: {dict(zip(loaded_folds, weights.tolist()))}')
        stacked = torch.stack(fold_probs_list)           # (F, N, 5)
        final_probs = (stacked * weights[:, None, None]).sum(dim=0)

    return final_probs  # (N, 5)

# 13. Run Inference for Both Models
Inference runs per-model. Each produces a `(N, 5)` probability matrix we'll combine in the next cell.

In [ ]:
effnet_probs   = None
convnext_probs = None

if ENSEMBLE_FLAGS['use_efficientnet']:
    print('\n' + '=' * 60)
    print('EfficientNet-B3 Inference')
    print('=' * 60)
    effnet_probs = run_inference_for_model(
        model_type='efficientnet',
        attention=ENSEMBLE_FLAGS['effnet_attention'],
        test_df=test_df,
        tta_transforms=effnet_tta,
        use_tta=ENSEMBLE_FLAGS['use_tta'],
    )
    print(f'  EfficientNet probs shape: {effnet_probs.shape}')

if ENSEMBLE_FLAGS['use_convnext']:
    print('\n' + '=' * 60)
    print('ConvNeXt-Small Inference')
    print('=' * 60)
    convnext_probs = run_inference_for_model(
        model_type='convnext',
        attention=ENSEMBLE_FLAGS['convnext_attention'],
        test_df=test_df,
        tta_transforms=convnext_tta,
        use_tta=ENSEMBLE_FLAGS['use_tta'],
    )
    print(f'  ConvNeXt probs shape: {convnext_probs.shape}')

print('\nInference complete.')

# 14. Ensemble Strategies
Five strategies implementing `ensemble_strategies.md`. Each strategy operates on the probability matrices from Cell 13 and returns a metrics dict.

- **Single-model fallback**: If only one model is enabled, all strategies degrade to that model's predictions (useful for sanity checks).
- **AUC-safe shape**: Hard-label strategies (majority_vote) construct a one-hot probability matrix so `roc_auc_score` still works.

In [ ]:
def evaluate_predictions(probs, true_labels, strategy_name):
    """Compute all metrics for a given probability matrix."""
    preds = probs.argmax(dim=1).numpy()
    acc  = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds, average='macro', zero_division=0)
    rec  = recall_score(true_labels, preds, average='macro', zero_division=0)
    f1   = f1_score(true_labels, preds, average='macro')
    try:
        # Normalize (AUC needs rows to sum to 1)
        probs_norm = probs.numpy().astype(np.float64)
        probs_norm = probs_norm / probs_norm.sum(axis=1, keepdims=True).clip(min=1e-12)
        roc_auc = roc_auc_score(true_labels, probs_norm, multi_class='ovr', average='macro')
    except ValueError:
        roc_auc = 0.0

    cm = confusion_matrix(true_labels, preds)

    print(f'\n{"=" * 60}')
    print(f'Strategy: {strategy_name}')
    print(f'{"=" * 60}')
    print(f'  Accuracy:  {acc  * 100:.2f}%')
    print(f'  Precision: {prec * 100:.2f}%')
    print(f'  Recall:    {rec  * 100:.2f}%')
    print(f'  F1 Score:  {f1   * 100:.2f}%')
    print(f'  ROC-AUC:   {roc_auc * 100:.2f}%')
    print('  Per-class recall:')
    for i in range(5):
        class_recall = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
        print(f'    C{i}: {class_recall * 100:.1f}%')
    print('  Confusion Matrix:')
    print(cm)

    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
            'F1': f1, 'AUC': roc_auc, 'CM': cm}


# ─────────────────────────────────────────────────────────────────────────────
# Strategy 1: Simple Averaging — equal weights
# Formula: p = 0.5 * p_effnet + 0.5 * p_convnext
# ─────────────────────────────────────────────────────────────────────────────
def strategy_simple_avg():
    if effnet_probs is not None and convnext_probs is not None:
        probs = 0.5 * effnet_probs + 0.5 * convnext_probs
        label = 'Simple Average (50/50)'
    elif effnet_probs is not None:
        probs, label = effnet_probs, 'EffNet Only (single)'
    else:
        probs, label = convnext_probs, 'ConvNeXt Only (single)'
    return evaluate_predictions(probs, true_labels, label)


# ─────────────────────────────────────────────────────────────────────────────
# Strategy 2: Weighted Averaging — tunable global weights
# Formula: p = w_conv * p_convnext + w_eff * p_effnet
# ─────────────────────────────────────────────────────────────────────────────
def strategy_weighted_avg():
    w_c = ENSEMBLE_FLAGS['convnext_weight']
    w_e = ENSEMBLE_FLAGS['efficientnet_weight']
    if effnet_probs is not None and convnext_probs is not None:
        probs = w_e * effnet_probs + w_c * convnext_probs
        label = f'Weighted Average (ConvNeXt={w_c}, EffNet={w_e})'
    elif effnet_probs is not None:
        probs, label = effnet_probs, 'EffNet Only (single)'
    else:
        probs, label = convnext_probs, 'ConvNeXt Only (single)'
    return evaluate_predictions(probs, true_labels, label)


# ─────────────────────────────────────────────────────────────────────────────
# Strategy 3: Per-Class Weighted — different weights for each class
# Formula: p[c] = w_conv[c] * p_convnext[c] + w_eff[c] * p_effnet[c]
# Since per-class weights sum to 1.0 per class, output probabilities
# still sum to 1.0 per row — no renormalization needed.
# ─────────────────────────────────────────────────────────────────────────────
def strategy_per_class_weighted():
    if effnet_probs is None or convnext_probs is None:
        print('Per-class weighted requires BOTH models — falling back to single.')
        probs = effnet_probs if effnet_probs is not None else convnext_probs
        return evaluate_predictions(probs, true_labels, 'Single Model (fallback)')

    w_c = torch.tensor(ENSEMBLE_FLAGS['per_class_weights']['convnext']).unsqueeze(0)      # (1, 5)
    w_e = torch.tensor(ENSEMBLE_FLAGS['per_class_weights']['efficientnet']).unsqueeze(0)  # (1, 5)
    probs = w_e * effnet_probs + w_c * convnext_probs                                     # (N, 5)
    # probs already sums to 1.0 per row since weights sum to 1.0 per class
    return evaluate_predictions(probs, true_labels, 'Per-Class Weighted Average')


# ─────────────────────────────────────────────────────────────────────────────
# Strategy 4: Majority Voting — hard-label aggregation
# Each fold of each model votes; majority wins; ConvNeXt breaks ties (better overall acc)
# ─────────────────────────────────────────────────────────────────────────────
def strategy_majority_vote():
    if effnet_probs is None or convnext_probs is None:
        print('Majority vote requires BOTH models — falling back to single.')
        probs = effnet_probs if effnet_probs is not None else convnext_probs
        return evaluate_predictions(probs, true_labels, 'Single Model (fallback)')

    eff_preds  = effnet_probs.argmax(dim=1)
    conv_preds = convnext_probs.argmax(dim=1)

    final_preds = []
    for e, c in zip(eff_preds, conv_preds):
        if e.item() == c.item():
            final_preds.append(e.item())
        else:
            # Tie: prefer ConvNeXt (higher overall accuracy)
            final_preds.append(c.item())

    # One-hot encode for consistent metric interface
    probs_onehot = torch.zeros_like(convnext_probs)
    probs_onehot.scatter_(1, torch.tensor(final_preds).unsqueeze(1), 1.0)
    # Blend in softmax average so AUC reflects ranking, not just hard preds
    probs_for_auc = 0.5 * probs_onehot + 0.5 * (0.5 * effnet_probs + 0.5 * convnext_probs)
    return evaluate_predictions(probs_for_auc, true_labels,
                                'Majority Vote (ConvNeXt tiebreak)')


# ─────────────────────────────────────────────────────────────────────────────
# Strategy 5: Rank-Based Fusion (Borda count)
# Each model ranks classes by probability; summed ranks determine winner
# Robust when models have different calibration (one over-confident, one under-confident)
# ─────────────────────────────────────────────────────────────────────────────
def strategy_rank_fusion():
    if effnet_probs is None or convnext_probs is None:
        print('Rank fusion requires BOTH models — falling back to single.')
        probs = effnet_probs if effnet_probs is not None else convnext_probs
        return evaluate_predictions(probs, true_labels, 'Single Model (fallback)')

    # Borda ranks: class with highest prob gets rank K-1, lowest gets 0
    # argsort().argsort() converts probabilities to rank positions in [0, K-1]
    eff_ranks  = effnet_probs.argsort(dim=1).argsort(dim=1).float()
    conv_ranks = convnext_probs.argsort(dim=1).argsort(dim=1).float()

    summed_ranks = eff_ranks + conv_ranks                           # (N, 5), values in [0, 2(K-1)]
    probs = summed_ranks / summed_ranks.sum(dim=1, keepdim=True).clip(min=1e-12)
    return evaluate_predictions(probs, true_labels, 'Borda Rank Fusion')


# ─────────────────────────────────────────────────────────────────────────────
# Strategy registry
# ─────────────────────────────────────────────────────────────────────────────
strategy_map = {
    'simple_avg':          strategy_simple_avg,
    'weighted_avg':        strategy_weighted_avg,
    'per_class_weighted':  strategy_per_class_weighted,
    'majority_vote':       strategy_majority_vote,
    'rank_fusion':         strategy_rank_fusion,
}
print('5 ensemble strategies defined:', list(strategy_map.keys()))

# 15. Run Selected Strategy (or All)

If `strategy='all'`, runs every strategy and collects results for comparison.
Otherwise runs just the selected one.

In [ ]:
true_labels = test_df['Label'].values
results = {}

if ENSEMBLE_FLAGS['strategy'] == 'all':
    print('\nRunning ALL ensemble strategies for comparison...')
    for name, func in strategy_map.items():
        results[name] = func()
else:
    selected = ENSEMBLE_FLAGS['strategy']
    print(f'\nRunning selected strategy: {selected}')
    results[selected] = strategy_map[selected]()

# 16. Strategy Comparison Summary

Side-by-side comparison of all strategies tested. Also flags the best strategy per metric and checks if the 80% accuracy target was met.

**Class 3 watchlist**: Exp logs show all solo models plateau at 11-16% C3 recall. Any strategy breaking 20% is a significant win.

In [ ]:
if len(results) > 1:
    print('\n' + '*' * 80)
    print('              ENSEMBLE STRATEGY COMPARISON')
    print('*' * 80)
    header = f"{'Strategy':<30} | {'Acc':>7} | {'Prec':>7} | {'Rec':>7} | {'F1':>7} | {'AUC':>7}"
    print(header)
    print('-' * 80)
    for name, res in results.items():
        print(f"{name:<30} | {res['Accuracy']*100:>6.2f}% | {res['Precision']*100:>6.2f}% | "
              f"{res['Recall']*100:>6.2f}% | {res['F1']*100:>6.2f}% | {res['AUC']*100:>6.2f}%")
    print('*' * 80)

    # ── Best-per-metric ──────────────────────────────────────────────────
    print('\nBEST STRATEGY PER METRIC:')
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']:
        best = max(results.items(), key=lambda x: x[1][metric])
        print(f'  {metric:>10}: {best[0]:<30} ({best[1][metric]*100:.2f}%)')

    # ── Class 3 watchlist ────────────────────────────────────────────────
    print('\nCLASS 3 (SEVERE DR) RECALL PER STRATEGY:')
    for name, res in results.items():
        cm = res['CM']
        c3_recall = cm[3, 3] / cm[3].sum() if cm[3].sum() > 0 else 0
        marker = ' ★' if c3_recall > 0.16 else ''
        print(f'  {name:<30} C3 recall: {c3_recall * 100:5.1f}%{marker}')

    # ── Target check ─────────────────────────────────────────────────────
    best_acc = max(r['Accuracy'] for r in results.values())
    if best_acc >= 0.80:
        best_name = max(results.items(), key=lambda x: x[1]['Accuracy'])[0]
        print(f'\n80% ACCURACY TARGET REACHED: {best_name} @ {best_acc*100:.2f}%')
    else:
        print(f'\n80% target not yet met. Best accuracy: {best_acc * 100:.2f}%')
        print('Consider: tuning weights, adding a 3rd model, or checking C3 data augmentation.')
else:
    print('Set ENSEMBLE_FLAGS["strategy"] = "all" to compare all 5 strategies side-by-side.')

# 17. Save Ensemble Results to JSON
Persists the full comparison to `ensemble_results/ensemble_<timestamp>.json` for later analysis or model card inclusion.

In [ ]:
from datetime import datetime

os.makedirs('ensemble_results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f'ensemble_results/ensemble_{timestamp}.json'

# Serialize confusion matrices as lists for JSON compatibility
serialized_results = {
    name: {
        'Accuracy':  float(res['Accuracy']),
        'Precision': float(res['Precision']),
        'Recall':    float(res['Recall']),
        'F1':        float(res['F1']),
        'AUC':       float(res['AUC']),
        'ConfusionMatrix': res['CM'].tolist(),
    }
    for name, res in results.items()
}

final_output = {
    'timestamp':       timestamp,
    'ensemble_flags':  ENSEMBLE_FLAGS,
    'model_paths':     MODEL_PATHS,
    'results':         serialized_results,
    'best_by_metric': {
        metric: max(serialized_results.items(), key=lambda x: x[1][metric])[0]
        for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
    } if len(results) > 1 else {},
}

with open(output_path, 'w') as f:
    json.dump(final_output, f, indent=2, default=str)

print(f'Results saved to: {output_path}')